# Fruit Classification — 03 · Modelling with TensorFlow/Keras

This notebook trains and compares two classifiers on the 24-class Fruits-360 subset:

1. a **small custom CNN** trained from scratch,
2. **VGG16 transfer learning** — ImageNet-pretrained convolutional base (frozen) with a new classification head.

Both models use the **leakage-aware split** from [02_preprocessing](02_preprocessing.ipynb) (`data/splits.csv`). The official dataset split interleaves near-identical video frames across train/val/test, so results on it are inflated; our metrics here are lower but honest. Augmentation is applied **to the training set only** — validation and test images are only resized and normalised.

> ⚡ Training is feasible on CPU but much faster on a GPU. On Colab: *Runtime → Change runtime type → GPU*.

## 1. Setup

In [ ]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not os.path.isdir("cnn_vgg16__fruit_classification"):
        !git clone https://github.com/Adriana394/cnn_vgg16__fruit_classification.git
    REPO_DIR = "cnn_vgg16__fruit_classification"
else:
    REPO_DIR = ".."  # this notebook lives in <repo>/notebooks/

DATA_ROOT = os.path.join(REPO_DIR, "data")
SPLITS_CSV = os.path.join(DATA_ROOT, "splits.csv")
print("Data root:", os.path.abspath(DATA_ROOT))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from PIL import Image
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.preprocessing.image import ImageDataGenerator

SEED = 42
IMG_SIZE = 100  # images come in varying sizes (see 01_eda) and are resized here
BATCH_SIZE = 64

tf.keras.utils.set_random_seed(SEED)
print("TensorFlow:", tf.__version__)
print("GPU available:", bool(tf.config.list_physical_devices("GPU")))

## 2. Data

We load `data/splits.csv` (created in 02_preprocessing) and build one DataFrame per split.

In [ ]:
splits = pd.read_csv(SPLITS_CSV)
train_df = splits[splits["split"] == "train"].reset_index(drop=True)
val_df = splits[splits["split"] == "val"].reset_index(drop=True)
test_df = splits[splits["split"] == "test"].reset_index(drop=True)

num_classes = splits["label"].nunique()
print(f"train: {len(train_df)}, val: {len(val_df)}, test: {len(test_df)}")
print(f"Number of classes: {num_classes}")

### Data generators

Both models share the same **augmentation** for training (rotations, shifts, shear, zoom and flips — all plausible for fruit photos on a white background). They differ only in **normalisation**:

- the small CNN uses simple `rescale=1/255`,
- VGG16 needs its own `preprocess_input` (the ImageNet preprocessing its pretrained weights were trained with).

Validation and test generators apply **no augmentation** — they only resize and normalise. (In the first version of this project the validation data was accidentally augmented and even drawn from the training folder via `validation_split`, which distorted the validation metrics.)

In [ ]:
AUGMENTATION = dict(
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode="nearest",
)


def make_generators(**normalisation):
    """Return (train, val, test) generators sharing one normalisation scheme.

    Augmentation is applied to the training generator only.
    """
    train_datagen = ImageDataGenerator(**AUGMENTATION, **normalisation)
    eval_datagen = ImageDataGenerator(**normalisation)

    def flow(datagen, frame, shuffle):
        return datagen.flow_from_dataframe(
            frame,
            directory=DATA_ROOT,
            x_col="filepath",
            y_col="label",
            target_size=(IMG_SIZE, IMG_SIZE),
            batch_size=BATCH_SIZE,
            class_mode="categorical",
            shuffle=shuffle,
            seed=SEED,
        )

    return (
        flow(train_datagen, train_df, shuffle=True),
        flow(eval_datagen, val_df, shuffle=False),
        flow(eval_datagen, test_df, shuffle=False),
    )

### Shared evaluation helpers

Used identically for both models: training curves, test metrics, confusion matrix, classification report and a few example predictions (displayed from the **raw** image files, since normalised arrays — especially after VGG16's `preprocess_input` — are not directly viewable).

In [ ]:
def plot_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history.history["loss"], label="train")
    axes[0].plot(history.history["val_loss"], label="validation")
    axes[0].set_title(f"{title} — loss")
    axes[0].set_xlabel("epoch")
    axes[0].legend()
    axes[1].plot(history.history["accuracy"], label="train")
    axes[1].plot(history.history["val_accuracy"], label="validation")
    axes[1].set_title(f"{title} — accuracy")
    axes[1].set_xlabel("epoch")
    axes[1].legend()
    plt.tight_layout()
    plt.show()


def evaluate_on_test(model, test_gen, name):
    """Evaluate on the test set; returns (loss, accuracy, predicted class indices)."""
    test_loss, test_acc = model.evaluate(test_gen, verbose=0)
    print(f"{name} — test accuracy: {test_acc:.4f} | test loss: {test_loss:.4f}")
    y_pred = model.predict(test_gen, verbose=0).argmax(axis=1)
    return test_loss, test_acc, y_pred


def class_names_of(gen):
    index_to_class = {v: k for k, v in gen.class_indices.items()}
    return [index_to_class[i] for i in range(len(index_to_class))]


def show_confusion_and_report(y_true, y_pred, class_names, name):
    fig, ax = plt.subplots(figsize=(11, 11))
    ConfusionMatrixDisplay(
        confusion_matrix(y_true, y_pred), display_labels=class_names
    ).plot(ax=ax, xticks_rotation="vertical", colorbar=False, values_format="d")
    ax.set_title(f"{name} — confusion matrix (test set)")
    plt.tight_layout()
    plt.show()
    print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))


def show_sample_predictions(y_pred, class_names, name, n=5):
    """Display raw test images with predicted vs. true label.

    Works because the test generator is built with shuffle=False, so
    prediction i corresponds to test_df row i.
    """
    rows = test_df.sample(n=n, random_state=SEED)
    fig, axes = plt.subplots(1, n, figsize=(3 * n, 3.5))
    for ax, (i, row) in zip(axes, rows.iterrows()):
        predicted = class_names[y_pred[i]]
        correct = predicted == row["label"]
        ax.imshow(Image.open(os.path.join(DATA_ROOT, row["filepath"])).resize((IMG_SIZE, IMG_SIZE)))
        ax.set_title(f"pred: {predicted}\ntrue: {row['label']}", color="green" if correct else "red", fontsize=9)
        ax.axis("off")
    plt.suptitle(f"{name} — sample test predictions")
    plt.tight_layout()
    plt.show()


results = {}  # collects test metrics for the final comparison

## 3. Small custom CNN

Four convolution blocks (Conv → BatchNorm → MaxPool → Dropout) with increasing filter counts, followed by a dense classifier. Trained from scratch with RMSprop.

Every epoch now iterates over the **full training set** (the first version used `steps_per_epoch=50`, which silently trained on only ~4% of the data per epoch). `EarlyStopping` restores the weights of the best validation epoch.

In [ ]:
train_gen_cnn, val_gen_cnn, test_gen_cnn = make_generators(rescale=1.0 / 255)

In [ ]:
def conv_block(filters, dropout):
    return [
        layers.Conv2D(filters, (3, 3), padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Dropout(dropout),
    ]


model_cnn = models.Sequential(
    [layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))]
    + conv_block(32, 0.2)
    + conv_block(64, 0.3)
    + conv_block(128, 0.4)
    + conv_block(256, 0.5)
    + [
        layers.Flatten(),
        layers.Dense(512, activation="relu"),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation="softmax"),
    ],
    name="small_cnn",
)

model_cnn.compile(
    loss="categorical_crossentropy",
    optimizer=optimizers.RMSprop(learning_rate=1e-4),
    metrics=["accuracy"],
)
model_cnn.summary()

In [ ]:
history_cnn = model_cnn.fit(
    train_gen_cnn,
    epochs=30,
    validation_data=val_gen_cnn,
    callbacks=[EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)],
)

In [ ]:
plot_history(history_cnn, "Small CNN")

### Evaluation on the test set

In [ ]:
class_names = class_names_of(test_gen_cnn)
y_true = test_gen_cnn.classes

cnn_loss, cnn_acc, y_pred_cnn = evaluate_on_test(model_cnn, test_gen_cnn, "Small CNN")
results["Small CNN"] = {"test accuracy": cnn_acc, "test loss": cnn_loss}

show_confusion_and_report(y_true, y_pred_cnn, class_names, "Small CNN")
show_sample_predictions(y_pred_cnn, class_names, "Small CNN")

## 4. VGG16 transfer learning

We load VGG16 with ImageNet weights, **without** its original classifier (`include_top=False`), and **freeze the convolutional base** — its general-purpose visual features stay fixed and only the new head (GlobalAveragePooling → Dense softmax) is trained. This is the standard transfer-learning recipe; it trains far fewer parameters and avoids destroying the pretrained features with large early gradients.

(The first version of this notebook set every VGG16 layer trainable, which amounts to full fine-tuning from the start — slower and riskier on a small dataset.)

*Optional next step — fine-tuning:* after the head has converged, unfreeze the last convolution block (`block5_*`) and continue training with a very low learning rate (e.g. `1e-5`).

The generators use VGG16's own `preprocess_input` instead of `rescale`.

In [ ]:
train_gen_vgg, val_gen_vgg, test_gen_vgg = make_generators(preprocessing_function=preprocess_input)

In [ ]:
vgg_base = VGG16(weights="imagenet", include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
vgg_base.trainable = False  # freeze the pretrained convolutional base

model_vgg = models.Sequential(
    [
        vgg_base,
        layers.GlobalAveragePooling2D(),
        layers.Dense(num_classes, activation="softmax"),
    ],
    name="vgg16_transfer",
)

model_vgg.compile(
    loss="categorical_crossentropy",
    optimizer=optimizers.Adam(learning_rate=1e-4),
    metrics=["accuracy"],
)
model_vgg.summary()

In [ ]:
history_vgg = model_vgg.fit(
    train_gen_vgg,
    epochs=10,
    validation_data=val_gen_vgg,
    callbacks=[EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)],
)

In [ ]:
plot_history(history_vgg, "VGG16 transfer")

### Evaluation on the test set

In [ ]:
vgg_loss, vgg_acc, y_pred_vgg = evaluate_on_test(model_vgg, test_gen_vgg, "VGG16 transfer")
results["VGG16 transfer"] = {"test accuracy": vgg_acc, "test loss": vgg_loss}

show_confusion_and_report(y_true, y_pred_vgg, class_names, "VGG16 transfer")
show_sample_predictions(y_pred_vgg, class_names, "VGG16 transfer")

## 5. Model comparison

In [ ]:
pd.DataFrame(results).T.round(4)

**How to read these numbers.** Both models are evaluated on the contiguous-block split, i.e. on frames the model has *not* seen near-duplicates of during training (apart from a few block-boundary frames). Scores are therefore lower than the near-perfect accuracies commonly reported on the official Fruits-360 split — and that is the point: those numbers measure memorisation, these measure generalisation to unseen views.

Keep in mind the limitation documented in 01_eda: each class is a single physical object, so this still tests *view* generalisation, not *object* generalisation.

Typical expectations: the frozen-base VGG16 converges in very few epochs because only its small classification head (~12k parameters) is trained on top of strong pretrained features, while the small CNN trains all of its parameters from scratch and needs more epochs. The confusion matrices usually show that mistakes concentrate among visually similar classes (e.g. the various red apple varieties).